# 🏀🌍 International basketball with `sportsdataverse-py`

The NBA is one league; the rest of the basketball world is dozens. `sportsdataverse-py` wraps the
feeds behind the biggest of them, and this notebook tours four:

| League | Module | Provider |
|---|---|---|
| EuroLeague | `sportsdataverse.euroleague` | EuroLeague Competition Engine |
| Super League Basketball (UK) | `sportsdataverse.basketball.genius` | Genius Sports LiveStats |
| NBL (Australia) | `sportsdataverse.basketball.nbl_au` | NBL Rosetta |
| Liga Endesa (Spain) | `sportsdataverse.basketball.acb` | ACB |

No API keys, polars by default (`return_as_pandas=True` for pandas), and at the end a EuroLeague
shot chart drawn on a FIBA court with [sdvplot](https://sdvplot.sportsdataverse.org/). 🎯

## 🧰 The toolbox

| Function | What it gives you |
|---|---|
| [`euroleague_seasons`](../euroleague/reference/euroleague.md#euroleague_seasons) | Seasons of a competition (`E` = EuroLeague, `U` = EuroCup) |
| [`euroleague_games`](../euroleague/reference/euroleague.md#euroleague_games) | Every game of a season, played or not |
| [`euroleague_standings`](../euroleague/reference/euroleague.md#euroleague_standings) | Standings after a round |
| [`euroleague_game_points`](../euroleague/reference/euroleague.md#euroleague_game_points) | Shot chart of one game (`coord_x` / `coord_y` in cm) |
| [`slb_competitions`](../genius/reference/additional.md#slb_competitions) | SLB competitions with their END-year season |
| [`slb_schedule`](../genius/reference/additional.md#slb_schedule) | Every match of an SLB season |
| [`slb_player_box`](../genius/reference/additional.md#slb_player_box) | Player box score of a finished SLB game |
| [`slb_shots`](../genius/reference/additional.md#slb_shots) | Shot chart of a finished SLB game |
| [`nbl_rosetta_standings`](../nbl_au/reference/nbl_rosetta.md#nbl_rosetta_standings) | NBL regular-season ladder |
| [`nbl_rosetta_matches`](../nbl_au/reference/nbl_rosetta.md#nbl_rosetta_matches) | Matches of an NBL season |
| [`acb_standings`](../acb/reference/acb.md#acb_standings) | Liga Endesa table after a round |
| [`acb_calendar`](../acb/reference/acb.md#acb_calendar) | Every round of a season with its games |

Genius LiveStats serves many leagues (`wbbl_*`, `bal_*`, `nznbl_*`, `esake_*`, `lfb_*`, ...) with the
same shape, so the SLB recipes below work for every one of them.

## 🔌 Setup

```sh
pip install "sportsdataverse[sdvplot]"
```

The `sdvplot` extra brings in sdvplot and sportypy for the court drawing.

In [ ]:
import matplotlib.pyplot as plt
import polars as pl

from sportsdataverse import euroleague as el
from sportsdataverse.basketball import acb, genius, nbl_au

pl.Config.set_tbl_rows(10)

Provider feeds are seasonal and occasionally rate-limited, so a tiny `safe()` helper runs each call
defensively: you get the frame when the feed is up, and a one-line skip when it isn't, naming whether
the answer was *nothing there* (`NoDataError`) or *the fetch failed* (`AssetFetchError`). 🛟

In [ ]:
from sportsdataverse.errors import AssetFetchError, NoDataError

def safe(label, thunk):
    try:
        out = thunk()
        print(f"✅ {label}")
        return out
    except (NoDataError, AssetFetchError) as e:
        print(f"\u23ed\ufe0f  {label}: {type(e).__name__}: {e}")
        return None

## 🇪🇺 EuroLeague

Season codes are the competition letter plus the START year (`E2025` is 2025-26). We take the newest
season that has a played game, so the notebook works in the off-season too.

In [ ]:
seasons = safe("EuroLeague seasons", lambda: el.euroleague_seasons("E"))
season_code, games = None, None
if seasons is not None:
    for code in seasons.sort("year", descending=True)["code"].head(3):
        g = safe(f"games {code}", lambda code=code: el.euroleague_games("E", code))
        if g is not None and g.filter(pl.col("played") == True).height:
            season_code, games = code, g
            break
print("season:", season_code, "| games:", None if games is None else games.shape)

In [ ]:
played = games.filter(pl.col("played") == True).sort("utc_date") if games is not None else None
cols = ["game_code", "round", "utc_date", "local_club_name", "road_club_name"]
(played.select([c for c in cols if c in played.columns]).tail(5)
 if played is not None and played.height else "no played games")

### Standings after the latest round 🏆

[`euroleague_standings`](../euroleague/reference/euroleague.md#euroleague_standings) takes the round as a string.

In [ ]:
last_round = str(played["round"].max()) if played is not None and played.height else None
stand = safe(f"standings round {last_round}",
             lambda: el.euroleague_standings("E", season_code, last_round)) if last_round else None
cols = ["position", "club_name", "games_played", "games_won", "games_lost", "points_difference"]
(stand.select([c for c in cols if c in stand.columns]).head(10)
 if stand is not None and stand.height else "standings unavailable")

### A shot chart on a FIBA court 🎯

[`euroleague_game_points`](../euroleague/reference/euroleague.md#euroleague_game_points) returns one row
per field-goal attempt and made free throw, in centimeters with the hoop at the origin.
`sdvplot.court_coords(..., provider="euroleague")` converts that frame to meters on the FIBA court
that `sdvplot.surface("fiba")` draws (free throws, encoded `-1, -1`, become null). Both teams shoot at
one basket, so we draw the defensive half.

In [ ]:
import sdvplot

gcode = str(played["game_code"][-1]) if played is not None and played.height else None
pts = safe(f"shots game {gcode}", lambda: el.euroleague_game_points(gcode, season_code)) if gcode else None
if pts is not None and pts.height:
    shots = sdvplot.court_coords(pts, x="coord_x", y="coord_y", provider="euroleague")
    shots = shots.filter(pl.col("court_x").is_not_null())
    made = shots.filter(pl.col("id_action").is_in(["2FGM", "3FGM"]))
    missed = shots.filter(pl.col("id_action").is_in(["2FGA", "3FGA"]))
    ax = sdvplot.surface("fiba", display_range="defense")
    ax.scatter(missed["court_x"], missed["court_y"], marker="x", s=30, c="#888888", zorder=20, label="missed")
    ax.scatter(made["court_x"], made["court_y"], s=30, c="#d62728", zorder=21, label="made")
    ax.legend(loc="upper right")
    ax.set_title(f"EuroLeague {season_code} game {gcode}: {shots.height} field-goal attempts")
    plt.show()
else:
    print("no shot data for this game")

## 🇬🇧 Super League Basketball (Genius Sports)

[`slb_competitions`](../genius/reference/additional.md#slb_competitions) lists the SLB competitions
Genius hosts; [`slb_schedule`](../genius/reference/additional.md#slb_schedule) defaults to the current one.

In [ ]:
comps = safe("SLB competitions", genius.slb_competitions)
comps.select([c for c in ["comp_id", "label", "season"] if c in comps.columns]).head() if comps is not None else None

In [ ]:
sched = safe("SLB schedule", genius.slb_schedule)
done = sched.filter(pl.col("status") == "COMPLETE") if sched is not None else None
cols = ["match_id", "date_time", "home_team_name", "home_score", "away_team_name", "away_score"]
(done.select([c for c in cols if c in done.columns]).tail(5)
 if done is not None and done.height else "no completed SLB games yet")

### One game: box score and shots 📋

Take the latest completed `match_id`, then the player box score
([`slb_player_box`](../genius/reference/additional.md#slb_player_box)) and the shot chart
([`slb_shots`](../genius/reference/additional.md#slb_shots), x/y as a percent of the full court).

In [ ]:
mid = int(done["match_id"][-1]) if done is not None and done.height else None
box = safe(f"SLB box {mid}", lambda: genius.slb_player_box(mid)) if mid else None
cols = ["name", "minutes", "points", "rebounds_total", "assists", "field_goals_made", "field_goals_attempted"]
(box.select([c for c in cols if c in box.columns]).sort("points", descending=True).head(8)
 if box is not None and box.height and "points" in box.columns else "box score unavailable")

In [ ]:
sh = safe(f"SLB shots {mid}", lambda: genius.slb_shots(mid)) if mid else None
(sh.group_by("action_type", "result").len().sort("action_type")
 if sh is not None and sh.height else "no shots for this game")

## 🇦🇺 NBL (Australia)

The NBL's Rosetta API takes the season as its END year.
[`nbl_rosetta_standings`](../nbl_au/reference/nbl_rosetta.md#nbl_rosetta_standings) gives the ladder;
[`nbl_rosetta_matches`](../nbl_au/reference/nbl_rosetta.md#nbl_rosetta_matches) the match list.

In [ ]:
nbl_season = 2026
ladder = safe(f"NBL ladder {nbl_season}", lambda: nbl_au.nbl_rosetta_standings(nbl_season))
cols = ["position", "team_name", "won", "lost", "win_percentage", "points_for", "points_against", "streak"]
(ladder.select([c for c in cols if c in ladder.columns]).sort("position")
 if ladder is not None and ladder.height else "ladder unavailable")

In [ ]:
matches = safe(f"NBL matches {nbl_season}", lambda: nbl_au.nbl_rosetta_matches(nbl_season))
(matches.group_by("match_status").len() if matches is not None and "match_status" in matches.columns
 else "matches unavailable")

## 🇪🇸 Liga Endesa (ACB)

[`acb_standings`](../acb/reference/acb.md#acb_standings) defaults to the table after the current round;
[`acb_calendar`](../acb/reference/acb.md#acb_calendar) lists every round of the season in one call.
A bar chart of the table makes the early-season race easy to read.

In [ ]:
table = safe("ACB standings", acb.acb_standings)
if table is not None and table.height:
    t = table.sort("win_pct", "plus_minus")
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.barh(t["team"], t["win_pct"], color="#1f4e79")
    ax.set_xlabel("win %")
    end = int(t["season"][0])  # END year
    ax.set_title(f"Liga Endesa {end - 1}-{str(end)[2:]} standings")
    plt.tight_layout()
    plt.show()
else:
    print("ACB standings unavailable")

In [ ]:
cal = safe("ACB calendar", acb.acb_calendar)
(cal.group_by("round_number").agg(games=pl.len(), played=pl.col("home_score").is_not_null().sum())
    .sort("round_number").head(8) if cal is not None and cal.height else "calendar unavailable")

## 🎉 Where to next

- Every Genius league (`wbbl_`, `bal_`, `nznbl_`, `esake_`, `lfb_`, `lnbm_`, `lub_`) has the same
  `_schedule` / `_player_box` / `_pbp` / `_shots` set as SLB.
- More homes under `sportsdataverse.basketball`: `fiba_hub`, `atrium`, `feb`, `scb`, `lba`, `vtb`,
  `kbl`, `bleague`, `cebl`, `aba` and others; each has a reference page in the sidebar.
- Shot charts: [sdvplot](https://sdvplot.sportsdataverse.org/)'s `court_coords` also converts NBA and
  WNBA stats-API shots (`provider="nba"`).
- NBA, WNBA and college basketball have their own notebooks (`04`, `08`, `06`, `05`).